<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.2-structured-output/practice/GCP_Capstone_3.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 3.2 — Structured Output & JSON Mode

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Run this first. Installs the unified `google-genai` SDK + Pydantic, authenticates with Application Default Credentials (no API keys), and initialises the Vertex client. Every exercise below depends on this cell.

In [ ]:
!pip install -q google-genai pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal, Optional, Union, List
import json, time

USD_INR = 85  # for any INR cost display

client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')
print('Client ready.')

## Exercise 1: First Pydantic Schema

**Difficulty:** Easy

Define a `Person` BaseModel (name, age, email, title) and extract from a free-form sentence. Call Gemini with `response_mime_type='application/json'` and `response_schema=Person`.

1. Define `Person(BaseModel)` with the four fields.
2. Call `client.models.generate_content` with `response_mime_type='application/json'` and `response_schema=Person`.
3. Read `r.parsed` (a typed `Person`) and confirm all four fields are populated and `r.text` has no markdown fences.

**Expected behaviour:** `r.parsed` is a `Person` instance. All four fields populated. No markdown fences in `r.text`.

In [ ]:
# Structured: parseable, typed, validated
class Person(BaseModel):
    name: str
    age: int
    email: str
    title: str

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Extract: John Doe, 32, john@ex.com, Senior Eng at Acme',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Person,
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
p: Person = r.parsed
print(f'type       : {type(p).__name__}')
print(f'name={p.name} | age={p.age} | email={p.email} | title={p.title}')
print(f'\nRaw text (no fences): {r.text[:120]}')

## Exercise 2: Field descriptions that change behaviour

**Difficulty:** Easy

Build `DocMetadata` with `Field(description=...)` on every field. Include at least one constraint ("max 120 chars") and one format ("ISO-8601 YYYY-MM-DD"). Compare outputs with and without the descriptions.

1. Define `DocMetadata` with rich `Field(description=...)` on each field.
2. Extract metadata from a sample document.
3. Define a bare copy with no descriptions and extract again; diff the two outputs.

**Expected behaviour:** With descriptions: titles capped, dates in ISO. Without: rambling titles, mixed date formats.

In [ ]:
# --- WITH descriptions ---
class DocMetadata(BaseModel):
    title: str = Field(description='Document title, max 120 chars')
    summary: str = Field(description='2-sentence abstract; no marketing language')
    topics: List[str] = Field(description='3-5 distinct topics, lowercase, no duplicates')
    language: Literal['en', 'hi', 'mixed'] = Field(description='Primary language')
    has_pii: bool = Field(description='True if document contains names, emails, phone, Aadhaar, PAN')
    confidence: Literal['high', 'medium', 'low']

text = '''Q4 2025 earnings: Revenue $12.3M (+18% YoY). CEO Priya Sharma noted strong India growth.
Contact: investor@acme.in, +91-98765-43210. PAN: ABCDE1234F.'''

def extract(schema):
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Extract document metadata:\n{text}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema,
            temperature=0.1,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.parsed

with_desc = extract(DocMetadata)
print('WITH descriptions:')
print(json.dumps(with_desc.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
# --- WITHOUT descriptions (same fields, bare) ---
class DocMetadataBare(BaseModel):
    title: str
    summary: str
    topics: List[str]
    language: Literal['en', 'hi', 'mixed']
    has_pii: bool
    confidence: Literal['high', 'medium', 'low']

no_desc = extract(DocMetadataBare)
print('WITHOUT descriptions:')
print(json.dumps(no_desc.model_dump(), indent=2, ensure_ascii=False))

print(f'\ntitle length  with={len(with_desc.title)}  without={len(no_desc.title)}')
print('The described version keeps the title terse and topics deduplicated;')
print('the bare version tends to ramble and mix formats.')

## Exercise 3: Enum classifier

**Difficulty:** Easy

Build a sentiment classifier with `response_mime_type='text/x.enum'` and enum `[POSITIVE, NEGATIVE, NEUTRAL]`. Run on 10 reviews. Measure latency vs a JSON-mode baseline.

1. Define an enum schema and a `classify_sentiment` helper using `text/x.enum`.
2. Build a JSON-mode baseline that returns the same label wrapped in an object.
3. Time both over 10 reviews and compare.

**Expected behaviour:** Enum mode is 20-40% faster. Output is plain text (not quoted JSON). All labels from the closed set.

In [ ]:
SENTIMENT_SCHEMA = {'type': 'STRING', 'enum': ['POSITIVE', 'NEGATIVE', 'NEUTRAL']}

def classify_enum(text):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f'Sentiment of this review: {text}',
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema=SENTIMENT_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.text

# JSON-mode baseline returning the same closed-set label
class Sentiment(BaseModel):
    label: Literal['POSITIVE', 'NEGATIVE', 'NEUTRAL']

def classify_json(text):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f'Sentiment of this review: {text}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=Sentiment,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.parsed.label

reviews = [
    'Best onboarding I have ever seen at a company.',
    'Wasted three hours debugging their broken SDK.',
    'The product exists. Nothing more to say.',
    'Support replied in five minutes and fixed it. Brilliant.',
    'Billing charged me twice and nobody responded.',
    'It works. Fine. Whatever.',
    'The dashboards are gorgeous and fast.',
    'Constant outages during our launch week.',
    'Docs are okay, could be better.',
    'Migration was painless and well documented.',
]

t0 = time.time()
enum_labels = [classify_enum(r) for r in reviews]
t_enum = time.time() - t0

t0 = time.time()
json_labels = [classify_json(r) for r in reviews]
t_json = time.time() - t0

for lab, rev in zip(enum_labels, reviews):
    print(f'  {lab:<8} | {rev}')
print(f'\nenum mode : {t_enum:.2f}s   json mode: {t_json:.2f}s   '
      f'({(1 - t_enum / t_json) * 100:.0f}% faster)')
assert all(l in {'POSITIVE', 'NEGATIVE', 'NEUTRAL'} for l in enum_labels)

## Exercise 4: Few-shot calibration for Indian invoices

**Difficulty:** Medium

Extract `Invoice` fields (vendor, invoice_number, total_inr, due_date ISO) from Indian-format inputs: "Rs 48,500/-", "3rd Feb 2026", "INV#442". Add 2-3 few-shot examples. Compare accuracy vs no examples over 10 inputs.

1. Define `Invoice` and a few-shot prompt with 2-3 worked examples.
2. Build a no-example (zero-shot) prompt for the same schema.
3. Run both over 10 Indian-format inputs and compare ISO-date / INR-float correctness.

**Expected behaviour:** With examples: 10/10 correct ISO dates, INR stripped to float. Without: ~6/10, mixed formats.

In [ ]:
class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    total_inr: float
    due_date: str = Field(description='ISO-8601 YYYY-MM-DD')

FEW_SHOT_PROMPT = '''Extract invoice fields. Follow the format shown.

Example:
Input: Bill from Cloudify Tech. Ref INV-2025-891. Rs. 48,500 due by 15/01/2026.
Output: {"vendor": "Cloudify Tech", "invoice_number": "INV-2025-891", "total_inr": 48500.0, "due_date": "2026-01-15"}

Example:
Input: Acme Corp INV#442 Rs 12,300 pay by 3rd Feb 2026
Output: {"vendor": "Acme Corp", "invoice_number": "INV#442", "total_inr": 12300.0, "due_date": "2026-02-03"}

Now extract:
Input: {input}
Output:'''

ZERO_SHOT_PROMPT = 'Extract invoice fields.\nInput: {input}\nOutput:'

def extract_invoice(prompt_tpl, inp):
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt_tpl.replace('{input}', inp),
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=Invoice,
            temperature=0.0,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.parsed

inputs = [
    'Invoice from BlueOcean Systems  Ref #BOS/2026/017  Rs 95,750/- payable 31 Mar 2026',
    'Bill: Tata Digital, INV-77, Rs 5,00,000 due 1st Jan 2027',
    'Zomato Ltd  #ZOM-9  Rs 1,299/-  by 12/12/2026',
    'From Reliance Retail INV#5521 Rs 48,500/- pay by 3rd Feb 2026',
    'Wipro Services ref WS-2026-88 Rs 2,45,000 due 28 Feb 2026',
    'Flipkart FK-441  Rs 899  by 5th Mar 2026',
    'Infosys BPM INV/2026/301 Rs 12,00,000/- payable 30/06/2026',
    'HDFC Ergo pol#HE-12 Rs 18,400 due 9th Apr 2026',
    'Swiggy Instamart #SW-3 Rs 640/- by 1 Jan 2026',
    'Ola Electric OLA-778 Rs 1,49,999 pay by 15th Aug 2026',
]

import re
def iso_ok(d):
    return bool(re.fullmatch(r'\d{4}-\d{2}-\d{2}', d))

few = [extract_invoice(FEW_SHOT_PROMPT, i) for i in inputs]
zero = [extract_invoice(ZERO_SHOT_PROMPT, i) for i in inputs]

few_ok = sum(iso_ok(inv.due_date) for inv in few)
zero_ok = sum(iso_ok(inv.due_date) for inv in zero)
print('few-shot sample:')
for inv in few[:3]:
    print(f'  {inv.vendor} | {inv.invoice_number} | Rs {inv.total_inr:,.0f} | due {inv.due_date}')
print(f'\nISO-date correct  few-shot={few_ok}/10   zero-shot={zero_ok}/10')

## Exercise 5: Nested RAGAnswer with citations

**Difficulty:** Medium

Build `RAGAnswer` with `citations: List[Citation]` where Citation = (chunk_id, quote). Feed a 3-chunk context and a question that spans two chunks. Verify both chunk IDs appear.

1. Define `Citation` and `RAGAnswer` (with `answer`, `citations`, `confidence`, `answerable`).
2. Provide a 3-chunk context and ask a question whose answer requires two chunks.
3. Assert both expected chunk IDs show up in the citations.

**Expected behaviour:** answer cites [1, 2] or similar. answerable=True. confidence high. Each quote is an exact substring of its chunk.

In [ ]:
class Citation(BaseModel):
    chunk_id: int
    quote: str = Field(description='Exact quote from source, max 200 chars')

class RAGAnswer(BaseModel):
    answer: str = Field(description='Answer grounded in the provided context')
    citations: List[Citation] = Field(description='All chunks that support the answer')
    confidence: Literal['high', 'medium', 'low']
    answerable: bool = Field(description='False if context does not contain the answer')

CONTEXT = '''[1] DocuMind supports PDF, DOCX, TXT, and Markdown files up to 200 MB.
[2] Uploads are processed via Doc AI Layout Parser v1.5 with 500-token chunks.
[3] Embeddings use text-embedding-005 at 768 dimensions.'''

# Question that spans chunk [1] (file types) and chunk [2] (chunking)
question = 'Which file types are supported, and how are the uploads chunked?'

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Answer only from the context.\n\nContext:\n{CONTEXT}\n\nQuestion: {question}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=RAGAnswer,
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
ans: RAGAnswer = r.parsed
print(f'Answerable: {ans.answerable} | Confidence: {ans.confidence}')
print(f'Answer: {ans.answer}')
cited = set()
for c in ans.citations:
    print(f'  [{c.chunk_id}] "{c.quote}"')
    cited.add(c.chunk_id)
print(f'\nchunk IDs cited: {sorted(cited)}')
assert {1, 2} <= cited, 'expected both chunks 1 and 2 to be cited'

## Exercise 6: Discriminated union for intent routing

**Difficulty:** Medium

Define three intents (search, summarize, translate) as discriminated Pydantic models on the `action` field. Classify 10 user utterances in ONE call each. Print the chosen variant and its required fields.

1. Define `SearchIntent`, `SummarizeIntent`, `TranslateIntent` on a `Literal` `action` discriminator.
2. Wrap them in `Intent` with `Union[...] = Field(discriminator='action')`.
3. Classify 10 utterances and print the chosen variant plus its populated fields.

**Expected behaviour:** Each utterance maps to exactly one variant. Required fields for the chosen variant are populated. No "impossible states" (translate without target_language).

In [ ]:
class SearchIntent(BaseModel):
    action: Literal['search']
    query: str
    top_k: int = 10

class SummarizeIntent(BaseModel):
    action: Literal['summarize']
    document_id: str
    length: Literal['short', 'medium', 'long']

class TranslateIntent(BaseModel):
    action: Literal['translate']
    text: str
    target_language: Literal['en', 'hi', 'ta', 'bn', 'te']

class Intent(BaseModel):
    intent: Union[SearchIntent, SummarizeIntent, TranslateIntent] = Field(discriminator='action')
    confidence: float = Field(ge=0.0, le=1.0)

utterances = [
    'Find me the last five deployment post-mortems.',
    'Give me a long summary of document doc-9f23.',
    'Translate this to Hindi: The deadline is next Monday.',
    'Search for onboarding checklists.',
    'Summarise doc-1122 in short.',
    'Convert this to Tamil: Welcome aboard.',
    'Look up the Q3 revenue report.',
    'Give a medium summary of doc-5580.',
    'Translate to Bengali: Please review the PR.',
    'Find all incidents tagged sev1.',
]
for u in utterances:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Classify user intent: {u}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=Intent,
            temperature=0.0,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    i: Intent = r.parsed
    fields = {k: v for k, v in i.intent.model_dump().items() if k != 'action'}
    print(f'{i.intent.action:<10} conf={i.confidence:.2f}  fields={fields}')

## Exercise 7: Truncation recovery

**Difficulty:** Challenge

Deliberately trigger `r.parsed is None` by setting `max_output_tokens=50` on a long extraction task. Implement a 3-level fallback: (1) retry with 4x tokens, (2) `schema.model_validate_json(r.text)` on the raw text, (3) return a sentinel error object. Log which path was taken.

1. Call with a tight `max_output_tokens` and detect `r.parsed is None`.
2. Path 1: retry with 4x the token budget.
3. Path 2: try `schema.model_validate_json(r.text)`; Path 3: return a sentinel with `finish_reason` logged.

**Expected behaviour:** On tight budget: path-1 retry succeeds. On SAFETY: paths 1-2 both fail, sentinel returned with finish_reason logged.

In [ ]:
def _call(prompt, schema, max_tokens):
    return client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema,
            temperature=0.0,
            max_output_tokens=max_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )

def _finish_reason(r):
    try:
        return str(r.candidates[0].finish_reason)
    except Exception:
        return 'UNKNOWN'

def safe_extract_3level(prompt, schema, max_tokens=50):
    # Attempt 0: tight budget (likely truncates -> parsed is None)
    r = _call(prompt, schema, max_tokens)
    if r.parsed is not None:
        return r.parsed, 'level-0-ok'

    # Level 1: retry with 4x tokens
    r = _call(prompt, schema, max_tokens * 4)
    if r.parsed is not None:
        return r.parsed, 'level-1-retry-4x'

    # Level 2: hand-parse the raw text
    try:
        return schema.model_validate_json(r.text), 'level-2-model_validate_json'
    except Exception:
        pass

    # Level 3: sentinel error object with finish_reason logged
    return {'error': 'unparseable', 'finish_reason': _finish_reason(r),
            'raw': (r.text or '')[:80]}, 'level-3-sentinel'

class PersonFull(BaseModel):
    name: str
    age: int
    email: str
    title: str
    bio: str = Field(description='A detailed multi-sentence biography')

prompt = ('Extract a full profile including a long detailed bio: '
          'John Doe, 32, john@ex.com, Senior Engineer at Acme, based in Bengaluru, '
          '10 years across search and ML platforms.')
val, path = safe_extract_3level(prompt, PersonFull, max_tokens=50)
print(f'path taken: {path}')
print(f'value: {val}')

## Exercise 8: Ship structured_output.py

**Difficulty:** Challenge

Package the six canonical schemas (RAGAnswer, DocMetadata, ExtractedEntity, ClassificationResult, EvalJudgement, Intent) + a `structured(prompt, schema_key)` helper into a single module. Write 6 unit tests. Confirm imports work from Modules 4-12 notebooks without copy-paste.

1. Write `structured_output.py` with the six schemas, a `SCHEMAS` registry, and a `structured(prompt, schema_key)` helper.
2. Write 6 unit tests (one per schema).
3. Import and call `structured` to confirm the module is usable elsewhere.

**Expected behaviour:** Single file structured_output.py. pytest green. Callable: `from structured_output import structured, RAGAnswer`.

In [ ]:
# Write the reusable module to disk so Modules 4-12 can `from structured_output import ...`
module_src = '''"""DocuMind structured_output: six canonical schemas reused across Modules 4-12."""
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal, Optional, Union, List


class Citation(BaseModel):
    chunk_id: int
    quote: str = Field(description='Exact quote from source, max 200 chars')

class RAGAnswer(BaseModel):
    answer: str
    citations: List[Citation]
    confidence: Literal['high', 'medium', 'low']
    answerable: bool

class DocMetadata(BaseModel):
    title: str
    summary: str
    topics: List[str]
    language: Literal['en', 'hi', 'mixed']
    has_pii: bool

class ExtractedEntity(BaseModel):
    kind: Literal['person', 'org', 'location', 'date', 'money', 'pii']
    text: str
    start: int
    end: int

class ClassificationResult(BaseModel):
    label: str
    confidence: float = Field(ge=0, le=1)
    reasoning: Optional[str] = None

class EvalJudgement(BaseModel):
    relevance: int = Field(ge=1, le=5)
    faithfulness: int = Field(ge=1, le=5)
    helpfulness: int = Field(ge=1, le=5)
    rationale: str

class SearchIntent(BaseModel):
    action: Literal['search']
    query: str
    top_k: int = 10

class SummarizeIntent(BaseModel):
    action: Literal['summarize']
    document_id: str
    length: Literal['short', 'medium', 'long']

class TranslateIntent(BaseModel):
    action: Literal['translate']
    text: str
    target_language: Literal['en', 'hi', 'ta', 'bn', 'te']

class Intent(BaseModel):
    intent: Union[SearchIntent, SummarizeIntent, TranslateIntent] = Field(discriminator='action')
    confidence: float = Field(ge=0.0, le=1.0)

SCHEMAS = {
    'rag': RAGAnswer,
    'doc_meta': DocMetadata,
    'entities': List[ExtractedEntity],
    'classify': ClassificationResult,
    'judge': EvalJudgement,
    'intent': Intent,
}


def structured(prompt, schema_key, project, location='us-central1',
               model='gemini-3.6-flash', temp=0.0):
    schema = SCHEMAS[schema_key]
    client = genai.Client(enterprise=True, project=project, location=location)
    r = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema, temperature=temp,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.parsed
'''

with open('structured_output.py', 'w') as f:
    f.write(module_src)
print('Wrote structured_output.py')

In [ ]:
# 6 unit tests — one per canonical schema. Validate construction + constraints (no API calls).
import importlib, structured_output
importlib.reload(structured_output)
from structured_output import (RAGAnswer, DocMetadata, ExtractedEntity,
                               ClassificationResult, EvalJudgement, Intent,
                               Citation, SCHEMAS)
from pydantic import ValidationError

def test_rag_answer():
    a = RAGAnswer(answer='x', citations=[Citation(chunk_id=1, quote='q')],
                  confidence='high', answerable=True)
    assert a.citations[0].chunk_id == 1

def test_doc_metadata():
    m = DocMetadata(title='t', summary='s', topics=['a'], language='en', has_pii=False)
    assert m.language == 'en'

def test_extracted_entity():
    e = ExtractedEntity(kind='person', text='Priya', start=0, end=5)
    assert e.end > e.start

def test_classification_result():
    c = ClassificationResult(label='spam', confidence=0.9)
    assert 0 <= c.confidence <= 1
    try:
        ClassificationResult(label='x', confidence=2.0); assert False
    except ValidationError:
        pass

def test_eval_judgement():
    j = EvalJudgement(relevance=5, faithfulness=4, helpfulness=3, rationale='ok')
    assert 1 <= j.relevance <= 5

def test_intent_discriminator():
    i = Intent(intent={'action': 'translate', 'text': 'hi', 'target_language': 'hi'},
               confidence=0.8)
    assert i.intent.action == 'translate'

for t in [test_rag_answer, test_doc_metadata, test_extracted_entity,
          test_classification_result, test_eval_judgement, test_intent_discriminator]:
    t()
    print(f'PASS {t.__name__}')
print(f'\n6/6 green. Registry keys: {list(SCHEMAS)}')

In [ ]:
# Confirm the helper is callable end-to-end (uses the module, not copy-pasted code).
from structured_output import structured

j = structured(
    'Rate this answer "Python is a snake." for the question "What is Python?"',
    'judge', project=PROJECT_ID)
print(f'Judge: rel={j.relevance} faith={j.faithfulness} help={j.helpfulness}')
print(f'Reason: {j.rationale}')
print('\nModule ready: from structured_output import structured, RAGAnswer')